#### ANÁLISE BRASIL

In [ ]:
# Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import os
from dbfread import DBF
import geopandas as gpd
import ssl
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

# Filtro de data
data_analise = '03/08/2026'
data_gal = '24/07/2026'

# Criar pasta para salvar os arquivos
nome_pasta = f'analises_chikungunya_Brasil'
local_salvar = os.getcwd()
local_pasta = os.path.join(local_salvar, nome_pasta)

if not os.path.exists(local_pasta):
    os.makedirs(local_pasta)
    print(f"Pasta {nome_pasta} criada!")
else:
    print(f"A pasta {nome_pasta} já existe. Os arquivos serão salvos nela.")

In [ ]:
# Processamento da base do SINAN
# 1. Os arquivos DBF serão convertidos para xlsx
# 2. Após a conversão serão automaticamente excluídos
# 3. Os arquivos xlsx serão concatenados em um arquivo único
# 4. Serão removidas colunas sem utilidade para análise
# 5. Serão criados os arquivos de casos notificados e prováveis
        # Os dados serão filtrados para um recorte de oito semanas, a partir da semana atual
# 6. Processamento da base do GAL
# 7. Transformação da tabela para um formato vertical e reorganização das métricas
        # Os dados serão filtrados para um recorte de oito semanas, a partir da semana atual

# CONVERSÃO DE DBF PARA XLSX E EXCLUSÃO DO DBF

# Obter o caminho absoluto do diretório atual
caminho_arquivos = os.path.abspath('')

# Listar todos os arquivos da pasta
arquivos_na_pasta = os.listdir(caminho_arquivos)

# Filtrar apenas arquivos que sejam .dbf (ou .DBF) e que comecem com 'CHIKON_'
arquivos_dbf = [
    arq for arq in arquivos_na_pasta 
    if arq.lower().endswith('.dbf') and arq.upper().startswith('CHIKON_')
]

if len(arquivos_dbf) > 0:
    print(f"Encontrados {len(arquivos_dbf)} arquivos DBF para converter.")
    
    for arquivo in arquivos_dbf:
        caminho_completo_dbf = os.path.join(caminho_arquivos, arquivo)
        
        # Cria o nome do novo arquivo trocando a extensão para .xlsx
        nome_sem_extensao = os.path.splitext(arquivo)[0]
        caminho_completo_xlsx = os.path.join(caminho_arquivos, f"{nome_sem_extensao}.xlsx")
        
        print(f"Convertendo: {arquivo}")
        
        try:
            # Lendo o DBF e convertendo para Excel
            tabela_dbf = DBF(caminho_completo_dbf, encoding='iso-8859-1')
            df_temp = pd.DataFrame(iter(tabela_dbf))
            df_temp.to_excel(caminho_completo_xlsx, index=False)
            
            # Se chegou até aqui sem dar erro, exclui o arquivo DBF original
            os.remove(caminho_completo_dbf)
            print(f"{nome_sem_extensao}.xlsx salvo com sucesso. Arquivo DBF excluído.")
            
        except Exception as e:
            print(f"Erro na conversão do arquivo {arquivo}. Motivo: {e}")
else:
    print("Nenhum arquivo DBF encontrado para conversão!\nVerificar a pasta!")

print("-" * 50)

# CONCATENAÇÃO ARQUIVOS XLSX E LIMPEZA DOS DADOS

# Atualizar a lista de arquivos na pasta
arquivos_atualizados = os.listdir(caminho_arquivos)

# Buscar todos os arquivos .xlsx que comecem com 'CHIKON_'
arquivos_excel = [
    arq for arq in arquivos_atualizados 
    if arq.lower().endswith('.xlsx') and arq.upper().startswith('CHIKON_')
]

if len(arquivos_excel) > 0:
    print(f"Concatenando {len(arquivos_excel)} arquivos")
    
    dfs = []
    
    # Ler cada arquivo e adicionar o DataFrame à lista
    for arquivo in arquivos_excel:
        caminho_completo = os.path.join(caminho_arquivos, arquivo)
        df_estados = pd.read_excel(caminho_completo)
        dfs.append(df_estados)
    
    # Concatenar todos os DataFrames em um único DataFrame
    df_notificados = pd.concat(dfs, ignore_index=True)

    print('Arquivos concatenados com sucesso!')
    
    # Remover colunas desnecessárias
    colunas_para_remover = [
        'SOUNDEX', 'ID_DISTRIT', 'ID_GEO1', 'ID_GEO2', 'DIABETES', 'HEMATOLOG', 'HEPATOPAT', 'RENAL', 'HIPERTENSA',
        'ACIDO_PEPT', 'AUTO_IMUNE', 'DT_CHIK_S1', 'DT_CHIK_S2', 'DT_PRNT', 'RES_CHIKS1', 'RES_CHIKS2', 'RESUL_PRNT', 
        'UF', 'MUNICIPIO', 'HOSPITAL', 'DDD_HOSP', 'TEL_HOSP', 'TPAUTOCTO', 'COUFINF', 
        'COPAISINF', 'COMUNINF', 'CODISINF', 'CO_BAINF', 'NOBAIINF', 'DOENCA_TRA', 'CLINC_CHIK', 'ALRM_HIPOT', 
        'ALRM_PLAQ', 'ALRM_VOM', 'ALRM_SANG', 'ALRM_HEMAT', 'ALRM_ABDOM', 'ALRM_LETAR', 'ALRM_HEPAT', 'ALRM_LIQ', 
        'DT_ALRM', 'GRAV_PULSO', 'GRAV_CONV', 'GRAV_ENCH', 'GRAV_INSUF', 'GRAV_TAQUI', 'GRAV_EXTRE', 'GRAV_HIPOT', 
        'GRAV_HEMAT', 'GRAV_MELEN', 'GRAV_METRO', 'GRAV_SANG', 'GRAV_AST', 'GRAV_MIOC', 'GRAV_CONSC', 'GRAV_ORGAO', 
        'DT_GRAV', 'MANI_HEMOR', 'EPISTAXE', 'GENGIVO', 'METRO', 'PETEQUIAS', 'HEMATURA', 'SANGRAM', 'LACO_N', 
        'PLASMATICO', 'EVIDENCIA', 'PLAQ_MENOR', 'CON_FHD', 'COMPLICA', 'NU_LOTE_I', 'NDUPLIC_N', 'DT_TRANSUS', 
        'DT_TRANSDM', 'DT_TRANSSM', 'DT_TRANSRM', 'DT_TRANSRS', 'DT_TRANSSE', 'NU_LOTE_V', 'NU_LOTE_H', 'CS_FLXRET', 
        'FLXRECEBI', 'IDENT_MICR', 'MIGRADO_W', 'SG_UF_NOT', 'ID_MUNICIP', 'ID_REGIONA', 'ID_UNIDADE', 'ID_PAIS', 'ID_OCUPA_N', 
        'DT_SORO', 'DT_NS1', 'DT_VIRAL', 'SOROTIPO', 'RESUL_VI_N', 'HISTOPA_N', 'IMUNOH_N', 'TP_SISTEMA', 'TP_NOT', 
        'CS_ESCOL_N', 'CS_GESTANT', 'DS_OBS', 'DT_ALARM', 'NM_LOGRADO', 'NU_LOTE_N', 'ID_LOGRADO', 'NM_REFEREN', 'NU_DDD_TEL', 'NU_TELEFON',
        'IN_VINCULA', 'FONETICA_N'
    ]
    
    colunas_existentes = [col for col in colunas_para_remover if col in df_notificados.columns]
    df_notificados = df_notificados.drop(columns=colunas_existentes)
    
    # Criação da coluna SE
    if 'SEM_PRI' in df_notificados.columns:
        posicao_sem_pri = df_notificados.columns.get_loc('SEM_PRI')    
        valores_se = df_notificados['SEM_PRI'].astype(str).str[-2:]
        df_notificados.insert(loc=posicao_sem_pri + 1, column='SE', value=valores_se)
    
    # CRIAR ARQUIVO DE CASOS NOTIFICADOS
    df_notificados.to_excel('chik_brasil_notificados_2026.xlsx', index=False)
    
    print('Arquivo com casos notificados criado com sucesso!')
     
    # CRIAR ARQUIVO DE CASOS PROVAVEIS
    
    # Filtragem do dataframe para retirar os casos descartados
    df_provaveis = df_notificados[df_notificados['CLASSI_FIN'] != 5]

    # Salvar data frame completo
    df_provaveis.to_excel('chik_brasil_provaveis_2026.xlsx', index= False)

    print('Arquivo com casos prováveis criado com sucesso!')

# ======================================================== #

# Processamento da base do GAL
# Carregar a aba mais detalhada (Municípios)
df_gal_bruto = pd.read_excel('chikungunya_analise_completa.xlsx', sheet_name='Chikungunya_por_Municipio')

# Desfazer (Melt) a tabela e transformar as dezenas de colunas em linhas
df_melt = df_gal_bruto.melt(
    id_vars=['paciente_municipio_ibge', 'paciente_municipio'], 
    var_name='Coluna_Original', 
    value_name='Valor'
)

# Extrair a Semana (SE) e o Tipo de Métrica (exames, positivos, etc)
# Exemplo: de '2026_SE01_exames' extrai '1' e 'exames'
df_melt['SE'] = df_melt['Coluna_Original'].str.extract(r'SE(\d+)').fillna(0)
df_melt['SE'] = df_melt['SE'].astype(int)
df_melt['Metrica'] = df_melt['Coluna_Original'].apply(lambda x: x.split('_')[-1])

# Reorganizar para ter as métricas como colunas arrumadas
df_gal_brasil = df_melt.pivot_table(
    index=['paciente_municipio_ibge', 'paciente_municipio', 'SE'],
    columns='Metrica',
    values='Valor',
    aggfunc='sum'
).reset_index()

# Limpeza e Cálculos Finais
# Preencher possíveis vazios com 0 e remover o nome do eixo de colunas
df_gal_brasil = df_gal_brasil.fillna(0)
df_gal_brasil.columns.name = None 

# Criar a coluna de código da UF para filtragem
df_gal_brasil['UF_COD'] = (df_gal_brasil['paciente_municipio_ibge']//10000).astype(int)
df_gal_brasil['UF_COD'] = pd.to_numeric(df_gal_brasil['UF_COD'], errors='coerce')

# Renomear as colunas para ficar elegante no seu código
df_gal_final = df_gal_brasil.rename(columns={
    'paciente_municipio_ibge': 'IBGE',
    'UF_COD': 'UF',
    'paciente_municipio': 'municipio',
    'exames': 'exames_realizados',
    'positivos': 'casos_positivos'
})

# Criar arquivo GAL completo
df_gal_final.to_excel('chik_brasil_gal_2026.xlsx')

print('Arquivo GAL criado com sucesso!')

# ======================================================== #

# criar lista com as bases de dados
bases_dados = [df_notificados, df_provaveis, df_gal_final]

# Função para garantir que a coluna SE será nuerica em todas as bases
for df_bases in bases_dados: 
    df_bases['SE'] = pd.to_numeric(df_bases['SE'], errors='coerce')

# Encontrar semana máxima das bases
se_final = max(df_bases['SE'].max() for df_bases in bases_dados)
se_atual = se_final - 1
semanas_analise = se_atual - 7

df_notificados_filtrado, df_provaveis_filtrado, df_gal_final_filtrado = [
    df_bases[df_bases['SE'].between(semanas_analise, se_atual)] for df_bases in bases_dados
]

print('Base de oito semanas criada com sucesso!')

# Mapeamento das unidades federativas por código IBGE
mapa_uf = {
    11: 'RO', 12: 'AC', 13: 'AM', 14: 'RR', 15: 'PA', 16: 'AP', 17: 'TO',
    21: 'MA', 22: 'PI', 23: 'CE', 24: 'RN', 25: 'PB', 26: 'PE', 27: 'AL', 28: 'SE', 29: 'BA',
    31: 'MG', 32: 'ES', 33: 'RJ', 35: 'SP', 41: 'PR', 42: 'SC', 43: 'RS', 50: 'MS', 51: 'MT', 
    52: 'GO', 53: 'DF'
}

# Carregar base dos municípios brasileiros
df_municipios = pd.read_excel('municipios_brasil.xlsx')

print('Base de municípios carregada com sucesso!')

# 1. MAPEAR, AGRUPAR E CONTAR OS CASOS POR UF
# Agrupa pela coluna SG_UF e conta a quantidade de registros (casos)
df_casos_uf = df_provaveis_filtrado['SG_UF'].map(mapa_uf).value_counts().reset_index()

# Renomeia as colunas
df_casos_uf.columns = ['Estado', 'Provaveis_2026']

# Ordena por número de casos
df_casos_uf = df_casos_uf.sort_values(by='Provaveis_2026').reset_index(drop=True)

print("Base de casos provaveis por UF criada com sucesso!")

print('-' * 50)
print('Todas as bases de dados foram criadas com sucesso!')

In [ ]:
# MAPA COROPLÉTICO DOS CASOS PROVÁVEIS POR ESTADO

# Mapear os códigos para as siglas dos estados
df_estados = df_provaveis['SG_UF'].map(mapa_uf).value_counts().reset_index()
df_estados.columns = ['Estado', 'Total']

# Desabilitar a verificação de SSL (caso tenha tido aquele problema)
ssl._create_default_https_context = ssl._create_unverified_context

# Carregar o mapa e cruzar os dados
mapa_brasil = gpd.read_file('brazil-states.geojson')
mapa_brasil = mapa_brasil.rename(columns={'sigla': 'Estado'})

mapa_casos = mapa_brasil.merge(df_estados, on='Estado', how='left')
mapa_casos['Total'] = mapa_casos['Total'].fillna(0) 

# CRIAR AS FAIXAS DE CORES
cortes = [-1, 0, 100, 500, 2000, 5000, float('inf')]

# Definir os textos que vão aparecer na legenda para cada faixa
rotulos = [
    'Sem casos', 
    '1 a 100 casos', 
    '101 a 500 casos', 
    '501 a 2000 casos', 
    '2001 a 5000 casos',
    '>5000 casos'
]

# Criar a coluna 'Categoria' dizendo em qual faixa cada estado se encaixa
mapa_casos['Categoria'] = pd.cut(mapa_casos['Total'], bins=cortes, labels=rotulos)

# Dicionário escolhendo a cor exata para cada texto da legenda
dicionario_cores = {
    'Sem casos': '#e9f5db',        
    '1 a 100 casos': '#cfe1b9',    
    '101 a 500 casos': '#b5c99a',  
    '501 a 2000 casos': '#97a97c',  
    '2001 a 5000 casos': '#87986a', 
    '>5000 casos': '#606c38'       
}

# Criar uma coluna aplicando a cor correspondente à categoria do estado
mapa_casos['Cor_Estado'] = mapa_casos['Categoria'].map(dicionario_cores)

# PLOTAR O MAPA
fig, ax = plt.subplots(figsize=(12, 12))

# Desenhar o mapa dizendo para usar exatamente a cor que colocamos na nova coluna
mapa_casos.plot(
    color=mapa_casos['Cor_Estado'], 
    linewidth=0.8, 
    edgecolor='0.3', # Linha cinza escura separando os estados
    ax=ax
)

# 5. CRIAR A LEGENDA CUSTOMIZADA
# Criamos "quadradinhos" (Patches) de cor para a legenda usando o nosso dicionário
lista_legendas = []
for rotulo in rotulos:
    quadradinho = mpatches.Patch(color=dicionario_cores[rotulo], label=rotulo)
    lista_legendas.append(quadradinho)

# Adicionar a legenda no mapa
ax.legend(
    handles=lista_legendas, 
    title='Casos Prováveis', 
    title_fontsize=14, 
    fontsize=12, 
    loc='lower right',
    bbox_to_anchor=(0.35, 0.17), # X (largura), Y (altura)
    frameon=True,
    shadow=False
)

# Estilização Final
ax.set_title(f'Distribuição Espacial de Casos Prováveis de Chikungunya por UF - 2026', 
             fontsize=18, fontweight='bold', pad=8)
ax.axis('off')
ax.text(
    0.13, 0.16,
    f'Fonte: SINAN\nDados até: {data_analise}',
    fontsize=10,
    color='black',
    transform=ax.transAxes,
    ha='left',
    va='top'
)

salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_mapa_uf.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# GRÁFICO DE CASOS PROVÁVEIS POR UNIDADE FEDERATIVA
# Transformar o código da coluna UF em números
df_provaveis['SG_UF'] = pd.to_numeric(df_provaveis['SG_UF'], errors='coerce')

# Mapear os códigos para as siglas dos estados
df_estados = df_provaveis['SG_UF'].map(mapa_uf).value_counts().reset_index()
df_estados.columns = ['Estado', 'Total']

# Configurar o estilo visual
grafico_estados = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_estados.add_subplot(gs[0])

# Criar o Gráfico de Barras Verticais
# Ordenado do maior para o menor automaticamente pelo value_counts()
sns.barplot(data=df_estados, x = 'Estado', y = 'Total', color= "#3a5a40", ax=ax)
ax.bar_label(ax.containers[0], fmt = '%d', padding = 3, fontsize = 10)
ax.set_title('Casos prováveis de chikungunya por UF - Brasil - 2026', 
          fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Estado Brasileiro', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Prováveis', fontsize=16, labelpad=16)
#ax.set_ylim (0, 16000)
ax.grid(axis='y', linestyle = '--', alpha = 0.7)
ax.grid(axis='x', linestyle = '--', alpha = 0.7)

# Informações rodapé 
ax_rodape = grafico_estados.add_subplot(gs[1])
ax_rodape.axis('OFF')
rodape_texto = f'Fonte: SINAN | Dados até: {data_analise}'
ax_rodape.text(0.0, 0.05, rodape_texto, ha = 'left', va = 'center', fontsize = 9, color = 'black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_uf.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches = 'tight')
plt.show()

In [ ]:
# GRÁFICO DE CASOS PROVÁVEIS POR UNIDADE FEDERATIVA NAS ÚLTIMAS OITO SEMANAS
# Transformar o código da coluna UF em números
df_provaveis_filtrado['SG_UF'] = pd.to_numeric(df_provaveis_filtrado['SG_UF'], errors='coerce')

# Mapear os códigos para as siglas dos estados
df_estados = df_provaveis_filtrado['SG_UF'].map(mapa_uf).value_counts().reset_index()
df_estados.columns = ['Estado', 'Total']

# Configurar o estilo visual
grafico_estados = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_estados.add_subplot(gs[0])

# Criar o Gráfico de Barras Verticais
# Ordenado do maior para o menor automaticamente pelo value_counts()
sns.barplot(data=df_estados, x = 'Estado', y = 'Total', color= "#fa9500", ax=ax)
ax.bar_label(ax.containers[0], fmt = '%d', padding = 3, fontsize = 10)
ax.set_title(f'Casos prováveis de chikungunya por UF - SE{semanas_analise} a SE{se_atual} - Brasil - 2026', 
          fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Estado Brasileiro', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Prováveis', fontsize=16, labelpad=16)
#ax.set_ylim (0, 16000)
ax.grid(axis='y', linestyle = '--', alpha = 0.7)
ax.grid(axis='x', linestyle = '--', alpha = 0.7)

# Informações rodapé 
ax_rodape = grafico_estados.add_subplot(gs[1])
ax_rodape.axis('OFF')
rodape_texto = f'Fonte: SINAN | Dados até: {data_analise}'
ax_rodape.text(0.0, 0.05, rodape_texto, ha = 'left', va = 'center', fontsize = 9, color = 'black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_uf_quatro_semanas.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches = 'tight')
plt.show()

In [ ]:
# MUNICÍPIOS COM MAIOR FREQUÊNCIA DE CASOS PROVÁVEIS
# Tratamento dos códigos IBGE para o cruzamento
# Transformar o código IBGE de 7 dígitos do arquivo em 6 dígitos (como no SINAN)
df_municipios['COD_IBGE_6'] = df_municipios['CÓDIGO DO MUNICÍPIO - IBGE'].astype(str).str[:6]

# Garantir que a coluna do SINAN também seja tratada como texto (string)
df_provaveis['ID_MN_RESI'] = df_provaveis['ID_MN_RESI'].astype(str)

# Cruzar (Merge) as duas bases
# Trazemos as informações do df_municipios para o df_sinan baseando-se no código de 6 dígitos
df_completo = pd.merge(df_provaveis, df_municipios, left_on='ID_MN_RESI', right_on='COD_IBGE_6', how='left')

# Contar os casos por município
# Contamos a frequência de cada nome na coluna 'MUNICÍPIO - IBGE'
casos_por_municipio = df_completo['MUNICÍPIO - IBGE - UF'].value_counts().reset_index()
casos_por_municipio.columns = ['Município', 'Total']

# Selecionar os 10 municípios com mais casos para o gráfico não ficar ilegível
top_municipios = casos_por_municipio.head(10)

# Configurar o estilo visual
grafico_municipio = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_municipio.add_subplot(gs[0])

# Criar o Gráfico de Barras Verticais
# Ordenado do maior para o menor automaticamente pelo value_counts()
sns.barplot(data=top_municipios, x = 'Município', y = 'Total', color= "#3a5a40", ax=ax)
ax.bar_label(ax.containers[0], fmt = '%d', padding = 3, fontsize = 10)
ax.set_title('Municípios com maior frequência de casos prováveis de chikungunya por UF - Brasil - 2026', 
          fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Município-UF', fontsize=16, labelpad=16)
ax.tick_params(axis='x', labelsize=10, rotation=15)
ax.set_ylabel('Casos Prováveis', fontsize=16, labelpad=16)
#ax.set_ylim (0, 10000)
ax.grid(axis='y', linestyle = '--', alpha = 0.7)
ax.grid(axis='x', linestyle = '--', alpha = 0.7)

# Informações rodapé 
ax_rodape = grafico_municipio.add_subplot(gs[1])
ax_rodape.axis('OFF')
rodape_texto = (f'Fonte: SINAN | Dados até: {data_analise}\n'
                f'Observação: a quantificação dos casos se deu a partir da variável município de residência (ID_MN_RESI)')
ax_rodape.text(0.0, -0.5, rodape_texto, ha = 'left', va = 'center', fontsize = 9, color = 'black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_municipios.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches = 'tight')
plt.show()

In [ ]:
# MUNICÍPIOS COM MAIOR FREQUÊNCIA DE CASOS PROVÁVEIS NAS ULTIMAS OITO SEMANAS
# Tratamento dos códigos IBGE para o cruzamento
# Transformar o código IBGE de 7 dígitos do arquivo em 6 dígitos (como no SINAN)
df_municipios['COD_IBGE_6'] = df_municipios['CÓDIGO DO MUNICÍPIO - IBGE'].astype(str).str[:6]

# Garantir que a coluna do SINAN também seja tratada como texto (string)
# IMPORTANTE: Substitua 'ID_MUNICIP' pelo nome exato da coluna de município na sua base do SINAN
df_provaveis_filtrado['ID_MN_RESI'] = df_provaveis_filtrado['ID_MN_RESI'].astype(str)

# Cruzar (Merge) as duas bases
# Trazemos as informações do df_municipios para o df_sinan baseando-se no código de 6 dígitos
df_completo = pd.merge(df_provaveis_filtrado, df_municipios, left_on='ID_MN_RESI', right_on='COD_IBGE_6', how='left')

# Contar os casos por município
# Contamos a frequência de cada nome na coluna 'MUNICÍPIO - IBGE'
casos_por_municipio = df_completo['MUNICÍPIO - IBGE - UF'].value_counts().reset_index()
casos_por_municipio.columns = ['Município', 'Total']

# Selecionar os 10 municípios com mais casos para o gráfico não ficar ilegível
top_municipios = casos_por_municipio.head(10)

# Configurar o estilo visual
grafico_municipio = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_municipio.add_subplot(gs[0])

# Criar o Gráfico de Barras Verticais
# Ordenado do maior para o menor automaticamente pelo value_counts()
sns.barplot(data=top_municipios, x = 'Município', y = 'Total', color= "#fa9500", ax=ax)
ax.bar_label(ax.containers[0], fmt = '%d', padding = 3, fontsize = 10)
ax.set_title(f'Municípios com maior frequência de casos prováveis de chikungunya por UF - SE{semanas_analise} a SE{se_atual} - Brasil - 2026', 
          fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Município-UF', fontsize=16, labelpad=16)
ax.tick_params(axis='x', labelsize=10, rotation=15)
ax.set_ylabel('Casos Prováveis', fontsize=16, labelpad=16)
#ax.set_ylim (0, 10000)
ax.grid(axis='y', linestyle = '--', alpha = 0.7)
ax.grid(axis='x', linestyle = '--', alpha = 0.7)

# Informações rodapé 
ax_rodape = grafico_municipio.add_subplot(gs[1])
ax_rodape.axis('OFF')
rodape_texto = (f'Fonte: SINAN | Dados até: {data_analise}\n'
                f'Observação: a quantificação dos casos se deu a partir da variável município de residência (ID_MN_RESI)')
ax_rodape.text(0.0, -0.5, rodape_texto, ha = 'left', va = 'center', fontsize = 9, color = 'black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_municipios_quatro_semanas.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches = 'tight')
plt.show()

In [ ]:
# MUNICIPIOS COM MAIOR FREQUENCIA DE CASOS PROVAVEIS EM OITO SEMANAS (COM SEGMENTAÇÃO DE QUATRO SEMANAS)

# TRATAMENTO E UNIAO DAS BASES
df_municipios['COD_IBGE_6'] = df_municipios['CÓDIGO DO MUNICÍPIO - IBGE'].astype(str).str[:6]
df_provaveis_filtrado['ID_MN_RESI'] = df_provaveis_filtrado['ID_MN_RESI'].astype(str)

df_completo = pd.merge(
    df_provaveis_filtrado, 
    df_municipios, 
    left_on='ID_MN_RESI', 
    right_on='COD_IBGE_6', 
    how='left'
)

# DEFINIÇÃO DOS DOIS PERÍODOS (PRIMEIRAS 4 X ÚLTIMAS 4 SEMANAS)
# Define o ponto de corte do primeiro bloco de 4 semanas
se_corte = semanas_analise + 3

# Rótulos para a legenda do gráfico
rotulo_p1 = f'SE{semanas_analise} a SE{se_corte}'
rotulo_p2 = f'SE{se_corte + 1} a SE{se_atual}'

df_completo['Período'] = np.where(
    df_completo['SE'] <= se_corte, 
    rotulo_p1, 
    rotulo_p2
)

top_10_municipios = df_completo['MUNICÍPIO - IBGE - UF'].value_counts().head(10).index

# Filtrar o DataFrame apenas com esses 10 municípios
df_top = df_completo[df_completo['MUNICÍPIO - IBGE - UF'].isin(top_10_municipios)]

# Agrupar contagem de casos por Município e Período
df_agrupado = df_top.groupby(['MUNICÍPIO - IBGE - UF', 'Período']).size().reset_index(name='Casos')

# Manter a ordem decrescente dos municípios com base no total de casos
df_agrupado['MUNICÍPIO - IBGE - UF'] = pd.Categorical(
    df_agrupado['MUNICÍPIO - IBGE - UF'], 
    categories=top_10_municipios, 
    ordered=True
)
df_agrupado = df_agrupado.sort_values('MUNICÍPIO - IBGE - UF')

# CONSTRUÇÃO DO GRÁFICO
grafico_municipio = plt.figure(figsize=(16, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_municipio.add_subplot(gs[0])

# Gráfico de Colunas Agrupadas (usando 'hue' para comparar os 2 períodos)
sns.barplot(data=df_agrupado, x='MUNICÍPIO - IBGE - UF', y='Casos', 
    hue='Período', palette=['#3a5a40', '#fa9500'],
    ax=ax)

# Adicionar rótulos com a quantidade em todas as colunas
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3, fontsize=9)

# Títulos e Eixos
ax.set_title(
    f'Comparativo de casos prováveis de Chikungunya - SE{semanas_analise} a SE{se_atual} - Brasil - 2026', 
    fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Município - UF', fontsize=16, labelpad=16)
ax.tick_params(axis='x', labelsize=10, rotation=15)  
ax.set_ylabel('Casos Prováveis', fontsize=16, labelpad=16)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.3)

# Legenda do Período
ax.legend(fontsize=11, loc='upper right', frameon=True)

# RODAPÉ E SALVAMENTO
ax_rodape = grafico_municipio.add_subplot(gs[1])
ax_rodape.axis('OFF')
rodape_texto = (
    f'Fonte: SINAN | Dados até: {data_analise}\n'
    f'Observação: A quantificação dos casos deu-se a partir da variável município de residência (ID_MN_RESI).'
)
ax_rodape.text(0.0, -0.5, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_municipios_comparativo_4_semanas.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# MUNICÍPIOS COM MAIOR FREQUÊNCIA DE CASOS CONFIRMADOS EM UM ESTADO NAS ÚLTIMAS OITO SEMANAS

# Tratamento dos códigos IBGE para o cruzamento
# Transformar o código IBGE de 7 dígitos do arquivo em 6 dígitos (como no SINAN)
df_municipios['COD_IBGE_6'] = df_municipios['CÓDIGO DO MUNICÍPIO - IBGE'].astype(str).str[:6]

# Garantir que a coluna do GAL também seja tratada como string
df_gal_final_filtrado['IBGE'] = df_gal_final_filtrado['IBGE'].astype(str)

# Cruzar (Merge) as duas bases
# Trazemos as informações do df_municipios para o df_sinan baseando-se no código de 6 dígitos
df_completo = pd.merge(df_gal_final_filtrado, df_municipios, left_on='IBGE', right_on='COD_IBGE_6', how='left')

# Contar os casos por município
# Contamos a frequência de cada nome na coluna 'MUNICÍPIO - IBGE'
casos_por_municipio = df_completo.groupby('MUNICÍPIO - IBGE - UF')['casos_positivos'].sum().reset_index()
casos_por_municipio = casos_por_municipio.sort_values(by='casos_positivos', ascending= False)
casos_por_municipio.columns = ['Município', 'Total']

# Selecionar os municípios com mais casos para o gráfico não ficar ilegível
top_municipios = casos_por_municipio.head(10)

# Configurar o estilo visual
grafico_municipio = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_municipio.add_subplot(gs[0])

# Criar o Gráfico de Barras Verticais
# Ordenado do maior para o menor automaticamente pelo value_counts()
sns.barplot(data=top_municipios, x = 'Município', y = 'Total', color= "#3a5a40", ax=ax)
ax.bar_label(ax.containers[0], fmt = '%d', padding = 3, fontsize = 10)
ax.set_title(f'Casos confirmados de chikungunya por municípios - Brasil - SE{semanas_analise} a SE{se_atual} - 2026', 
          fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Município', fontsize=16, labelpad=16)
ax.tick_params(axis='x', labelsize=10, rotation=15)
ax.set_ylabel('Casos Confirmados', fontsize=16, labelpad=16)
#ax.set_ylim (0, 10000)
ax.grid(axis='y', linestyle = '--', alpha = 0.7)
ax.grid(axis='x', linestyle = '--', alpha = 0.7)

# Informações rodapé 
ax_rodape = grafico_municipio.add_subplot(gs[1])
ax_rodape.axis('OFF')
rodape_texto = (f'Fonte: GAL | Dados até: {data_gal}\n')
ax_rodape.text(0.0, -0.5, rodape_texto, ha = 'left', va = 'center', fontsize = 9, color = 'black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_confirmados_municipios_quatro_semanas.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches = 'tight')
plt.show()

In [ ]:
# CASOS PROVÁVEIS E CONFIRMADOS (GAL) POR SEMANA EPIDEMIOLÓGICA
# Carregamento e Tratamento dos Dados
df_semanas = df_provaveis.groupby('SE').size().reset_index(name='Total')
df_gal_semana = df_gal_final.groupby('SE')[['casos_positivos']].sum().reset_index()

# União de dois dataframe 
# Isso garante que a barra e a linha fiquem perfeitamente alinhadas na mesma semana
df_merged = pd.merge(df_semanas, df_gal_semana, on='SE', how='left').fillna(0)

# Configurar o estilo visual
grafico_sem = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_sem.add_subplot(gs[0])

# Criar o Gráfico de Barras Verticais (Usando 'color' e 'label')
sns.barplot(data=df_merged, x='SE', y='Total', color="#3a5a40", ax=ax, label='Prováveis (SINAN)')

# Rótulos das Barras
ax.bar_label(ax.containers[0], fmt='%d', padding=3, fontsize=10)

# Criar o Gráfico de Linha
# ax.plot é mais seguro para garantir o alinhamento central com as barras do seaborn
ax.plot(df_merged.index, df_merged['casos_positivos'], color="#fa9500", linewidth=3, marker='o', markersize=6, label='Confirmados (GAL)')

# Rótulos da Linha (Loop que passa por cada ponto e escreve o número em cima)
for i, row in df_merged.iterrows():
    # Só coloca o rótulo se tiver mais de 0 casos para não poluir o gráfico
    if row['casos_positivos'] > 0: 
        # O +80 abaixo empurra o texto um pouco para cima do ponto da linha para não sobrepor
        ax.text(i, row['casos_positivos'] + 80, f"{int(row['casos_positivos'])}", 
                ha='center', va='bottom', fontsize=9, color="#dad7cd", fontweight='bold', rotation=0)

# Estilização do Eixo e Legenda
ax.set_title('Casos prováveis e confirmados de chikungunya por semana epidemiológica - Brasil - 2026', 
            fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Prováveis e Confirmados', fontsize=16, labelpad=16)
#ax.set_xlim(len(df_gal_final['SE']))
#ax.set_ylim(0, 2700)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)

# Agora a legenda vai funcionar perfeitamente pegando os 'labels' definidos nas barras e linhas
ax.legend(fontsize=11, loc='upper right', frameon=True)

# Informações rodapé 
ax_rodape = grafico_sem.add_subplot(gs[1])
ax_rodape.axis('off')
rodape_texto = f'Fonte: SINAN e GAL | Dados até: {data_analise}\n'
# Coloquei 0.5 no eixo Y para centralizar direitinho no rodapé
ax_rodape.text(0.0, 0.0, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_confirmados_semana.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# COMPARATIVO ENTRE CONFIRMADOS SINAN E CONFIRMADOS GAL
# Filtrar casos confirmados na base do SINAN
df_semanas = df_notificados[df_notificados['CLASSI_FIN'] == 13]['SE'].value_counts().reset_index()
df_semanas.columns = ['SE', 'Total']
df_semanas = df_semanas.sort_values('SE').reset_index(drop=True)

# Contar os casos positivos na base do GAL
df_conf = df_gal_final.groupby('SE')['casos_positivos'].sum().reset_index()
df_conf.columns = ['SE', 'Total']

df_semanas['Confirmados'] = 'Confirmados (SINAN)'
df_conf['Confirmados'] = 'Confirmados (GAL)'

# Configurar o estilo visual
grafico_sem = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_sem.add_subplot(gs[0])

df_combinado = pd.concat([df_semanas, df_conf], ignore_index=True)

# Criar o Gráfico de Barras Verticais
sns.barplot(data=df_combinado, x='SE', y='Total', hue='Confirmados',
            palette= ['#3a5a40', '#fa9500'], ax=ax, width= 0.9)

for c in ax.containers:
    ax.bar_label(c, fmt='%d', padding=3, fontsize=10)

# Estilização dos Eixos e Legenda
ax.set_title('Casos confirmados de chikungunya por SE - GAL e SINAN - Brasil - 2026', 
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Confirmados', fontsize=16, labelpad=16)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)
ax.legend(title='', fontsize=11, title_fontsize=12, loc='upper right', frameon=True)

# Informações rodapé 
ax_rodape = grafico_sem.add_subplot(gs[1])
ax_rodape.axis('off')
rodape_texto = (f'Fonte: SINAN e GAL | Dados até: {data_analise} | '
                f'Total de confirmados SINAN: {df_semanas['Total'].sum():.0f} | '
                f'Total de confirmados GAL: {df_conf['Total'].sum():.0f}')

ax_rodape.text(0.0, 0.0, rodape_texto, ha='left', va='center', fontsize=10, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_confirmados_sinan_gal.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# COMPARATIVO ENTRE EXAMES REALIZADOS E CONFIRMADOS GAL POR SEMANA EPIDEMIOLÓGICA
# contar exames realizados na base do GAL
df_exames = df_gal_final.groupby('SE')['exames_realizados'].sum().reset_index()
df_exames.columns = ['SE', 'Total']

# Contar os casos positivos na base do GAL
df_conf = df_gal_final.groupby('SE')['casos_positivos'].sum().reset_index()
df_conf.columns = ['SE', 'Total']

df_exames['Exames'] = 'Exames Realizados'
df_conf['Exames'] = 'Exames Confirmados'

# Configurar o estilo visual
grafico_sem = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_sem.add_subplot(gs[0])

df_gal_combinado = pd.concat([df_exames, df_conf], ignore_index=True)

# Criar o Gráfico de Barras Verticais
sns.barplot(data=df_gal_combinado, x='SE', y='Total', hue='Exames',
            palette= ['#3a5a40', '#fa9500'], ax=ax, width= 0.9)

for c in ax.containers:
    ax.bar_label(c, fmt='%d', padding=3, fontsize=10)

# Estilização dos Eixos e Legenda
ax.set_title('Exames realizados e positivos GAL para chikungunya por SE - Brasil - 2026', 
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Exames Realizados e Confirmados', fontsize=16, labelpad=16)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)
ax.legend(title='', fontsize=11, title_fontsize=12, loc='upper right', frameon=True)

# Informações rodapé 
ax_rodape = grafico_sem.add_subplot(gs[1])
ax_rodape.axis('off')
rodape_texto = (f'Fonte: GAL | Dados até: {data_gal} | '
                f'Total de Exames Realizados: {df_exames['Total'].sum():.0f} | '
                f'Total de Exames Confirmados: {df_conf['Total'].sum():.0f}')

ax_rodape.text(0.0, 0.0, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_realizados_confirmados_gal_semana.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# CASOS CONFIRMADOS POR CRITÉRIO DE CONFIRMAÇÃO APLICADO
# Filtrar casos conformados por critério laboratorial na base do SINAN
df_conf_lab = df_notificados[(df_notificados['CLASSI_FIN'] == 13) & (df_notificados['CRITERIO'] == 1)]['SE'].value_counts().reset_index()
df_conf_lab.columns = ['SE', 'Total']
df_conf_lab = df_conf_lab.sort_values('SE').reset_index(drop=True)

# Filtrar casos confirmados por criterio clini-epidemiologico na base do SINAN
df_conf_clep = df_notificados[(df_notificados['CLASSI_FIN'] == 13) & (df_notificados['CRITERIO'] == 2)]['SE'].value_counts().reset_index()
df_conf_clep.columns = ['SE', 'Total']
df_conf_clep = df_conf_clep.sort_values('SE').reset_index(drop=True)

df_conf_lab['Confirmados'] = 'Laboratorial (SINAN)'
df_conf_clep['Confirmados'] = 'Clinico-Epidemiológico'

# Configurar o estilo visual
grafico_sem = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_sem.add_subplot(gs[0])

df_combinado = pd.concat([df_conf_lab, df_conf_clep], ignore_index=True)

# Criar o Gráfico de Barras Verticais
sns.barplot(data=df_combinado, x='SE', y='Total', hue='Confirmados',
            palette= ['#3a5a40', '#fa9500'], ax=ax, width= 0.9)

for c in ax.containers:
    ax.bar_label(c, fmt='%d', padding=3, fontsize=10)

# Estilização dos Eixos e Legenda
ax.set_title('Casos confirmados de chikungunya por SE e critério de confirmação - Brasil - 2026', 
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Confirmados', fontsize=16, labelpad=16)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)
ax.legend(title='', fontsize=11, title_fontsize=12, loc='upper right', frameon=True)

# Informações rodapé 
ax_rodape = grafico_sem.add_subplot(gs[1])
ax_rodape.axis('off')
rodape_texto = (f'Fonte: SINAN | Dados até: {data_analise} | '
                f'Confirmados - Laboratorial: {df_conf_lab['Total'].sum():.0f} | '
                f'Confirmados - Clinico-Epidemiológico: {df_conf_clep['Total'].sum():.0f}')

ax_rodape.text(0.0, 0.0, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_confirmados_criterio.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Tratamento da Classificação
# Transformar em número. O que for vazio ('') vira NaN (Nulo)
df_notificados['CLASSI_FIN'] = pd.to_numeric(df_notificados['CLASSI_FIN'], errors='coerce')

# Preencher os valores nulos (NaN) com um código numérico fictício (ex: 99) 
# Isso impede que o groupby ignore os casos "Em Investigação"
df_notificados['CLASSI_FIN'] = df_notificados['CLASSI_FIN'].fillna(99)

# Novo dicionário com o código 99 representando o que estava vazio
mapa_class = {5: 'Descartado', 13: 'Confirmado', 99: 'Em Investigação'}

# Agrupar os dados
# Dataframe para o gráfico de barras empilhadas e tabela
df_class = df_notificados.groupby(['SE', 'CLASSI_FIN']).size().unstack(fill_value=0) 
df_class = df_class.rename(columns=mapa_class)

# Configurar o estilo visual
grafico_class = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_class.add_subplot(gs[0])

# Criar o Gráfico de Barras Empilhadas
# Adicionei cores distintas para cada classificação
cores = {'Confirmado': '#3a5a40', 'Descartado': '#fa9500', 'Em Investigação': '#eb5e28'}
df_class.plot(kind='bar', stacked=True, ax=ax, width=0.8, color=[cores.get(col, '#333333') for col in df_class.columns])

# Adicionar os rótulos de dados (bar_label)
# Como são barras empilhadas, precisamos passar por CADA container (cada pedaço da barra)
for c in ax.containers:
    # Condição para só escrever o número se ele for maior que 0 (evita poluir o gráfico com zeros)
    labels = [f'{v.get_height():.0f}' if v.get_height() > 0 else '' for v in c]
    # label_type='center' coloca o número bem no meio da cor respectiva
    ax.bar_label(c, labels=labels, label_type='center', fontsize=9, color='white', fontweight='bold')

ax.set_title('Classificação final dos casos notificados de chikungunya por semana epidemiológica - Brasil - 2026', 
          fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Notificados', fontsize=16, labelpad=16)

# O eixo X no pandas fica rotacionado em 90 graus por padrão. Isso conserta e deixa reto:
plt.xticks(rotation=0) 

# Ajustar o limite do eixo Y (adapte se necessário)
# ax.set_ylim(0, 1400) # Removi temporariamente para o pandas ajustar automaticamente às barras somadas

# Melhorar a legenda
ax.legend(title='Classificação Final', fontsize=11, title_fontsize=12, loc='upper right')

ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7) # Tirar as linhas verticais para ficar mais limpo

# Informações rodapé 
ax_rodape = grafico_class.add_subplot(gs[1])
ax_rodape.axis('off') # Minúsculo
rodape_texto = f'Fonte: SINAN | Dados até: {data_analise}'
# Ajustado para ficar centralizado verticalmente no rodapé
ax_rodape.text(0.0, 0.5, rodape_texto, ha='left', va='center', fontsize=10, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_provaveis_classificacao.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# BASE DE POPULAÇÃO E MAPEAMENTO DE REGIÕES (IBGE)
# População aproximada por UF (Essencial para calcular o Coeficiente de Incidência)
dados_uf_pop = {
    'Estado': ['RO', 'AC', 'AM', 'RR', 'PA', 'AP', 'TO', 
              'MA', 'PI', 'CE', 'RN', 'PB', 'PE', 'AL', 'SE', 'BA',
              'MG', 'ES', 'RJ', 'SP', 
              'PR', 'SC', 'RS',
              'MS', 'MT', 'GO', 'DF'],
    'Nome_UF': ['Rondônia', 'Acre', 'Amazonas', 'Roraima', 'Pará', 'Amapá', 'Tocantins',
                'Maranhão', 'Piauí', 'Ceará', 'Rio Grande do Norte', 'Paraíba', 'Pernambuco', 
                'Alagoas', 'Sergipe', 'Bahia', 'Minas Gerais', 'Espírito Santo', 'Rio de Janeiro', 
                'São Paulo', 'Paraná', 'Santa Catarina', 'Rio Grande do Sul', 'Mato Grosso do Sul', 
                'Mato Grosso', 'Goiás', 'Distrito Federal'],
    'Regiao': ['Norte']*7 + ['Nordeste']*9 + ['Sudeste']*4 + ['Sul']*3 + ['Centro-Oeste']*4,
    'Populacao': [1581196, 830018, 3941613, 636707, 8120131, 733759, 1511460, # Norte
                 6776805, 3270174, 8794957, 3303953, 3974687, 9058931, 3125254, 2210003, 14141626, # Nordeste
                 20539989, 3833712, 16055174, 44411238, # Sudeste
                 11444380, 7610361, 10882965, # Sul
                 2757013, 3658649, 7056495, 2817381]} # Centro-Oeste

df_pop = pd.DataFrame(dados_uf_pop)

# PROCESSAMENTO E CRUZAMENTO DOS DADOS
# Ler os arquivos Excel salvos anteriormente
df_chik_25 = pd.read_excel('chik_brasil_provaveis_2025_por_uf.xlsx')
df_chik_26 = pd.read_excel('chik_brasil_provaveis_2026_por_uf.xlsx')

# Garantir que os nomes das colunas de casos fiquem padronizados
df_chik_25.columns = ['Estado', 'Provaveis_2025']
df_chik_26.columns = ['Estado', 'Provaveis_2026']

# Juntar os dados de 2025 e 2026 com a tabela de População
df_casos_anos = pd.merge(df_chik_25, df_chik_26, on='Estado', how='outer').fillna(0)
df_consolidado = pd.merge(df_pop, df_casos_anos, on='Estado', how='left').fillna(0)

# Calcular Incidência por 100 mil habitantes
df_consolidado['Inc_2026'] = (df_consolidado['Provaveis_2026'] / df_consolidado['Populacao']) * 100000
df_consolidado['Inc_2025'] = (df_consolidado['Provaveis_2025'] / df_consolidado['Populacao']) * 100000

# MONTAGEM DA HIERARQUIA (SUBTOTAIS POR REGIÃO)
linhas_tabela = []

# Agrupar por Região respeitando a ordem geográfica
ordem_regioes = ['Norte', 'Nordeste', 'Sudeste', 'Sul', 'Centro-Oeste']

for reg in ordem_regioes:
    df_reg = df_consolidado[df_consolidado['Regiao'] == reg]
    if df_reg.empty:
        continue
        
    # Subtotal da Região
    pop_reg = df_reg['Populacao'].sum()
    c26_reg = df_reg['Provaveis_2026'].sum()
    c25_reg = df_reg['Provaveis_2025'].sum()
    inc26_reg = (c26_reg / pop_reg) * 100000 if pop_reg > 0 else 0
    inc25_reg = (c25_reg / pop_reg) * 100000 if pop_reg > 0 else 0
    var_reg = ((c26_reg - c25_reg) / c25_reg * 100) if c25_reg > 0 else (100.0 if c26_reg > 0 else 0.0)
    
    # Adiciona a Linha da Região (Subtotal)
    linhas_tabela.append({
        'Local': reg, 'is_regiao': True,
        'Provaveis_2026': c26_reg, 'Inc_2026': inc26_reg,
        'Provaveis_2025': c25_reg, 'Inc_2025': inc25_reg,
        'Var': var_reg
    })
    
    # Adiciona as Linhas de cada Estado da Região
    for _, row in df_reg.iterrows():
        c26 = row['Provaveis_2026']
        c25 = row['Provaveis_2025']
        var_uf = ((c26 - c25) / c25 * 100) if c25 > 0 else (100.0 if c26 > 0 else 0.0)
        
        linhas_tabela.append({
            'Local': f"  {row['Nome_UF']}", 'is_regiao': False,
            'Provaveis_2026': c26, 'Inc_2026': row['Inc_2026'],
            'Provaveis_2025': c25, 'Inc_2025': row['Inc_2025'],
            'Var': var_uf
        })

df_report = pd.DataFrame(linhas_tabela)

# FORMATAÇÃO TEXTUAL E MONTAGEM DA MATRIZ DA TABELA
# Formatadores
fmt_int = lambda x: f"{int(x):,}".replace(',', '.')
fmt_dec = lambda x: f"{x:,.1f}".replace(',', 'X').replace('.', ',').replace('X', '.')

# Função para formatar decimais com as setas ▲ e ▼
def fmt_dec_setas(val):
    if pd.isna(val) or val == 0:
        return "0,0"
    
    # Formata com uma casa decimal no padrão BR
    valor_abs_str = f"{abs(val):,.0f}".replace(',', 'X').replace('.', ',').replace('X', '.')
    
    if val > 0:
        return f"▲ {valor_abs_str}"
    else:
        return f"▼ {valor_abs_str}"

# Construir células de texto
celulas_texto = []
# Linhas de Cabeçalho Superior e Inferior
#celulas_texto.append(['', '', '', 'CASOS PROVÁVEIS', '', ''])
celulas_texto.append(['REGIÃO/UF', 'Casos 2026', 'INCIDÊNCIA', 'Casos 2025', 'INCIDÊNCIA', '% de variação'])

for _, row in df_report.iterrows():
    celulas_texto.append([
        row['Local'],
        fmt_int(row['Provaveis_2026']),
        fmt_dec(row['Inc_2026']),  # Função decimal normal para incidência
        fmt_int(row['Provaveis_2025']),
        fmt_dec(row['Inc_2025']),  # Função decimal normal para incidência
        fmt_dec_setas(row['Var'])  # <--- NOVA FUNÇÃO COM SETAS PARA A VARIAÇÃO
    ])

# RENDERIZAÇÃO VISUAL COM MATPLOTLIB
fig, ax = plt.subplots(figsize=(11, len(celulas_texto) * 0.38 + 1))
ax.axis('tight')
ax.axis('off')

tabla = ax.table(
    cellText=celulas_texto,
    cellLoc='center',
    loc='center'
)

tabla.auto_set_font_size(False)
tabla.set_fontsize(11)
tabla.scale(1.2, 1.5)

# --- MESCLAGEM DE CÉLULAS DO CABEÇALHO ---
#tabla[(0, 3)].get_text().set_text('CASOS PROVÁVEIS')
# Ajustar tamanhos de cabeçalho
for col in range(1, 6):
    tabla[(0, col)].set_facecolor('#d9d9d9')

tabla[(0, 1)].get_text().set_text('PROVAVEIS 2026')
tabla[(0, 3)].get_text().set_text('PROVAVEIS 2025')
tabla[(0, 5)].get_text().set_text('VARIAÇÃO (%)')

# Estilização Célula por Célula
n_cabecalho_offset = 1  # linhas de cabeçalho

for (row, col), cell in tabla.get_celld().items():
    cell.set_linewidth(0.6)
    cell.set_edgecolor('#a0a0a0')
    
    # Estilo dos Cabeçalhos
    if row < n_cabecalho_offset:
        cell.set_facecolor('#cccccc')
        cell.set_text_props(weight='bold', color='black')
        if col == 0:
            cell.set_facecolor('#d9d9d9')
    else:
        idx_data = row - n_cabecalho_offset
        info_linha = df_report.iloc[idx_data]
        
        # Alinhar nomes das UFs/Regiões à esquerda
        if col == 0:
            cell.set_text_props(ha='left')
        
        # Destaque de LINHA DE REGIÃO (SUBTOTAL)
        if info_linha['is_regiao']:
            cell.set_facecolor('#d9d9d9')
            cell.set_text_props(weight='bold')
        else:
            cell.set_facecolor('#ffffff')
            
        # Destaque Condicional: Variação positiva (Aumento de casos) em VERMELHO
        if col == 5:
            val_var = info_linha['Var']
            if val_var > 0:
                cell.set_facecolor('#fce4d6')  # Fundo levemente alaranjado/rosado
                cell.set_text_props(color='#c00000', weight='bold') # Texto vermelho escuro

# RODAPÉ
texto_rodape = (
    "Fonte: SINAN - Sistema de Informação de Agravos de Notificação\n"
    "Nota: Incidência calculada por 100.000 habitantes com base na população estimada (IBGE)."
)

plt.figtext(
    0.05,        # Posição Horizontal (0.05 = margem esquerda)
    0.18,       # Posição Vertical (valores negativos colocam o texto abaixo da tabela)
    texto_rodape,
    ha='left',   # Alinhamento horizontal à esquerda
    va='top',    # Alinhamento vertical no topo do texto
    fontsize=9,
    style='italic',
    color="#000000")

plt.title(
    'Casos Prováveis de Chikungunya e Coeficiente de Incidência por Região e UF - Brasil - 2025 e 2026', 
    fontsize=15, weight='bold', pad= -60
)

# Salvar tabela
salvar_pasta = os.path.join(local_pasta, 'chik_brasil_tabela_comparativa_uf.png')
plt.savefig(salvar_pasta , bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# SALVAR GRÁFICOS EM PDF ÚNICO
def gerar_pdf_brasil(lista_arquivos, nome_pdf='Relatorio_Chikungunya_Brasil_2026.pdf'):
    c = canvas.Canvas(nome_pdf, pagesize=A4)
    largura, altura = A4
    margem = 30
    
    # Função auxiliar para desenhar o cabeçalho em todas as páginas
    def desenhar_cabecalho(canvas):
        canvas.setFont("Helvetica-Bold", 16)
        canvas.drawCentredString(largura / 2, altura - 25, "Análise Chikungunya - Brasil - 2026")
        canvas.line(margem, altura - 35, largura - margem, altura - 35) # Linha divisória

    def processar_paginas(imagens_lista):
        # Definimos um espaçamento interno entre as imagens
        espacamento = 5 
        area_h = (altura / 2) - (margem + 40) 
        largura_max = largura - (2 * margem)
    
        def desenhar_figura(img_path, pos_y):
            # Verifica se a imagem existe antes de desenhar (evita o erro OSError)
            if os.path.exists(img_path):
                c.drawImage(img_path, margem, pos_y, width=largura_max, height=area_h, preserveAspectRatio=True)
            else:
                print(f"⚠️ Aviso: A imagem {img_path} não foi encontrada.")
        
        def desenhar_rodape(canvas, texto):
            canvas.setFont("Helvetica-Oblique", 5) 
    
            # 1. Obtém as dimensões atuais da página do canvas (largura e altura)
            largura_pagina, altura_pagina = canvas._pagesize
    
            # 2. Calcula o ponto central do eixo X
            centro_x = largura_pagina / 2.0
    
            # 3. Usa o método específico para desenhar texto centralizado
            # O eixo Y continua sendo 20 (próximo à borda inferior)
            canvas.drawCentredString(centro_x, 20, texto)

        texto_rodape = 'Análise realizada por Coordenação-Geral de Vigilância das Arboviroses (CGARB/DEDT/SVSA/MS)'

        # 1. PÁGINA 1
        desenhar_cabecalho(c)
        # Ajustei o Y para dar uma margem extra em relação ao cabeçalho e ao centro
        desenhar_figura(imagens_lista[0], (altura / 2) + espacamento) 
        desenhar_figura(imagens_lista[1], margem)
        c.showPage()
    
        # 2. PÁGINAS SEGUINTES
        restante = imagens_lista[2:]
        for i in range(0, len(restante), 2):
            desenhar_cabecalho(c)
            grupo = restante[i:i+2]
        
            pos_y_lista = [(altura / 2) + espacamento, margem]
        
            for idx, img_path in enumerate(grupo):
                desenhar_figura(img_path, pos_y_lista[idx])
            
            desenhar_rodape(c, texto_rodape)
            c.showPage()

    processar_paginas(lista_arquivos)
    c.save()
    print(f"Arquivo {os.path.basename(nome_pdf)} gerado com sucesso!")

# Lista dos arquivos
arquivos = [
    'chik_brasil_provaveis_mapa_uf.png',
    'chik_brasil_tabela_comparativa_uf.png',
    'chik_brasil_provaveis_uf.png',
    'chik_brasil_provaveis_uf_quatro_semanas.png',
    'chik_brasil_provaveis_municipios.png',
    'chik_brasil_provaveis_municipios_quatro_semanas.png',
    'chik_brasil_provaveis_municipios_comparativo_4_semanas.png',
    'chik_brasil_confirmados_municipios_quatro_semanas.png',
    'chik_brasil_provaveis_confirmados_semana.png',
    'chik_brasil_confirmados_sinan_gal.png',
    'chik_brasil_confirmados_criterio.png',
    'chik_brasil_provaveis_classificacao.png',
]

graficos = [os.path.join(local_pasta, nome) for nome in arquivos]
salvar_pdf = os.path.join(local_pasta, f'Relatorio_Chikungunya_Brasil_2026.pdf')

gerar_pdf_brasil(graficos, nome_pdf=salvar_pdf)

#### CHIKUNGUNYA - ESTADOS BRASILEIROS

In [ ]:
# DICIONÁRIO DE SIGLAS DOS ESTADOS
# 11: 'RO', 12: 'AC', 13: 'AM', 14: 'RR', 15: 'PA', 16: 'AP', 17: 'TO',
# 21: 'MA', 22: 'PI', 23: 'CE', 24: 'RN', 25: 'PB', 26: 'PE', 27: 'AL', 
# 28: 'SE', 29: 'BA', 31: 'MG', 32: 'ES', 33: 'RJ', 35: 'SP', 41: 'PR', 
# 42: 'SC', 43: 'RS', 50: 'MS', 51: 'MT', 52: 'GO', 53: 'DF'

# FILTROS PARA DELIMITAR ANÁLISE
nome_estado = 'Bahia'
filtro_estado = 29
data_analise = '03/08/2026'

# ======================================================== #

# Criar pasta para salvar os arquivos
nome_pasta = f'analises_chikungunya_{nome_estado}'
local_salvar = os.getcwd()
local_pasta = os.path.join(local_salvar, nome_pasta)

if not os.path.exists(local_pasta):
    os.makedirs(local_pasta)
    print(f"Pasta {nome_pasta} criada!")
else:
    print(f"A pasta {nome_pasta} já existe. Os arquivos serão salvos nela.")

In [ ]:
# MUNICÍPIOS COM MAIOR FREQUÊNCIA DE CASOS PROVÁVEIS EM UM ESTADO - ACUMULADO 2026

# Filtrar a base do SINAN para o estado desejado
df_sinan = df_provaveis[df_provaveis['SG_UF'] == filtro_estado]

# Tratamento dos códigos IBGE para o cruzamento
# Transformar o código IBGE de 7 dígitos do arquivo em 6 dígitos (como no SINAN)
df_municipios['COD_IBGE_6'] = df_municipios['CÓDIGO DO MUNICÍPIO - IBGE'].astype(str).str[:6]

# Garantir que a coluna do SINAN também seja tratada como string
df_sinan['ID_MN_RESI'] = df_sinan['ID_MN_RESI'].astype(str)

# Cruzar (Merge) as duas bases
# Trazemos as informações do df_municipios para o df_sinan baseando-se no código de 6 dígitos
df_completo = pd.merge(df_sinan, df_municipios, left_on='ID_MN_RESI', right_on='COD_IBGE_6', how='left')

# Contar os casos por município
# Contamos a frequência de cada nome na coluna 'MUNICÍPIO - IBGE'
casos_por_municipio = df_completo['MUNICÍPIO - IBGE'].value_counts().reset_index()
casos_por_municipio.columns = ['Município', 'Total']

# Selecionar os municípios com mais casos para o gráfico não ficar ilegível
top_municipios = casos_por_municipio.head(10)

# Configurar o estilo visual
grafico_municipio = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_municipio.add_subplot(gs[0])

# Criar o Gráfico de Barras Verticais
# Ordenado do maior para o menor automaticamente pelo value_counts()
sns.barplot(data=top_municipios, x = 'Município', y = 'Total', color= "#326273", ax=ax)
ax.bar_label(ax.containers[0], fmt = '%d', padding = 3, fontsize = 10)
ax.set_title(f'Casos prováveis de chikungunya por municípios - {nome_estado} - 2026', 
          fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Município', fontsize=16, labelpad=16)
ax.tick_params(axis='x', labelsize=10, rotation=15)
ax.set_ylabel('Casos Prováveis', fontsize=16, labelpad=16)
#ax.set_ylim (0, 10000)
ax.grid(axis='y', linestyle = '--', alpha = 0.7)
ax.grid(axis='x', linestyle = '--', alpha = 0.7)

# Informações rodapé 
ax_rodape = grafico_municipio.add_subplot(gs[1])
ax_rodape.axis('OFF')
rodape_texto = (f'Fonte: SINAN | Dados até: {data_analise}\n'
                f'Observação: a quantificação dos casos se deu a partir da variável município de residência (ID_MN_RESI)')
ax_rodape.text(0.0, -0.9, rodape_texto, ha = 'left', va = 'center', fontsize = 9, color = 'black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, f'chik_{nome_estado}_provaveis_municipios.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches = 'tight')
plt.show()

In [ ]:
# MUNICÍPIOS COM MAIOR FREQUÊNCIA DE CASOS PROVÁVEIS EM UM ESTADO NAS ÚLTIMAS OITO SEMANAS

# Filtrar a base do SINAN para o estado desejado
df_sinan_mun = df_provaveis_filtrado[df_provaveis_filtrado['SG_UF'] == filtro_estado]

# Tratamento dos códigos IBGE para o cruzamento
# Transformar o código IBGE de 7 dígitos do arquivo em 6 dígitos (como no SINAN)
df_municipios['COD_IBGE_6'] = df_municipios['CÓDIGO DO MUNICÍPIO - IBGE'].astype(str).str[:6]

# Garantir que a coluna do SINAN também seja tratada como string
df_sinan_mun['ID_MN_RESI'] = df_sinan_mun['ID_MN_RESI'].astype(str)

# Cruzar (Merge) as duas bases
# Trazemos as informações do df_municipios para o df_sinan baseando-se no código de 6 dígitos
df_completo = pd.merge(df_sinan_mun, df_municipios, left_on='ID_MN_RESI', right_on='COD_IBGE_6', how='left')

# Contar os casos por município
# Contamos a frequência de cada nome na coluna 'MUNICÍPIO - IBGE'
casos_por_municipio = df_completo['MUNICÍPIO - IBGE'].value_counts().reset_index()
casos_por_municipio.columns = ['Município', 'Total']

# Selecionar os municípios com mais casos para o gráfico não ficar ilegível
top_municipios = casos_por_municipio.head(10)

# Configurar o estilo visual
grafico_municipio = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_municipio.add_subplot(gs[0])

# Criar o Gráfico de Barras Verticais
# Ordenado do maior para o menor automaticamente pelo value_counts()
sns.barplot(data=top_municipios, x = 'Município', y = 'Total', color= "#eb5e28", ax=ax)
ax.bar_label(ax.containers[0], fmt = '%d', padding = 3, fontsize = 10)
ax.set_title(f'Casos prováveis de chikungunya por municípios - {nome_estado} - SE{semanas_analise} a SE{se_atual} - 2026', 
          fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Município', fontsize=16, labelpad=16)
ax.tick_params(axis='x', labelsize=10, rotation=15)
ax.set_ylabel('Casos Prováveis', fontsize=16, labelpad=16)
#ax.set_ylim (0, 10000)
ax.grid(axis='y', linestyle = '--', alpha = 0.7)
ax.grid(axis='x', linestyle = '--', alpha = 0.7)

# Informações rodapé 
ax_rodape = grafico_municipio.add_subplot(gs[1])
ax_rodape.axis('OFF')
rodape_texto = (f'Fonte: SINAN | Dados até: {data_analise}\n'
                f'Observação: a quantificação dos casos se deu a partir da variável município de residência (ID_MN_RESI)')
ax_rodape.text(0.0, -0.9, rodape_texto, ha = 'left', va = 'center', fontsize = 9, color = 'black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, f'chik_{nome_estado}_provaveis_municipios_quatro_semanas.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches = 'tight')
plt.show()

In [ ]:
# MUNICIPIOS COM MAIOR FREQUENCIA DE CASOS PROVAVEIS EM OITO SEMANAS (COM SEGMENTAÇÃO DE QUATRO SEMANAS)

# Filtrar a base do SINAN para o estado desejado
df_sinan_mn = df_provaveis_filtrado[df_provaveis_filtrado['SG_UF'] == filtro_estado]

# TRATAMENTO E UNIAO DAS BASES
df_municipios['COD_IBGE_6'] = df_municipios['CÓDIGO DO MUNICÍPIO - IBGE'].astype(str).str[:6]
df_sinan_mn['ID_MN_RESI'] = df_sinan_mn['ID_MN_RESI'].astype(str)

df_completo = pd.merge(
    df_sinan_mn, 
    df_municipios, 
    left_on='ID_MN_RESI', 
    right_on='COD_IBGE_6', 
    how='left'
)

# DEFINIÇÃO DOS DOIS PERÍODOS (PRIMEIRAS 4 X ÚLTIMAS 4 SEMANAS)
# Define o ponto de corte do primeiro bloco de 4 semanas
se_corte = semanas_analise + 3

# Rótulos para a legenda do gráfico
rotulo_estado_p1 = f'SE{semanas_analise} a SE{se_corte}'
rotulo_estado_p2 = f'SE{se_corte + 1} a SE{se_atual}'

df_completo['Período'] = np.where(
    df_completo['SE'] <= se_corte, 
    rotulo_estado_p1, 
    rotulo_estado_p2
)

top_10_municipios = df_completo['MUNICÍPIO - IBGE'].value_counts().head(10).index

# Filtrar o DataFrame apenas com esses 10 municípios
df_top = df_completo[df_completo['MUNICÍPIO - IBGE'].isin(top_10_municipios)]

# Agrupar contagem de casos por Município e Período
df_agrupado = df_top.groupby(['MUNICÍPIO - IBGE', 'Período']).size().reset_index(name='Casos')

# Manter a ordem decrescente dos municípios com base no total de casos
df_agrupado['Período'] = pd.Categorical(
    df_agrupado['Período'], 
    categories=[rotulo_estado_p1, rotulo_estado_p2], 
    ordered=True
)

# Manter a ordem decrescente dos municípios com base no total de casos
df_agrupado['MUNICÍPIO - IBGE'] = pd.Categorical(
    df_agrupado['MUNICÍPIO - IBGE'], 
    categories=top_10_municipios, 
    ordered=True
)

df_agrupado = df_agrupado.sort_values(['MUNICÍPIO - IBGE', 'Período'])

# CONSTRUÇÃO DO GRÁFICO
grafico_municipio = plt.figure(figsize=(16, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_municipio.add_subplot(gs[0])

# Gráfico de Colunas Agrupadas (usando 'hue' para comparar os 2 períodos)
sns.barplot(
    data=df_agrupado, 
    x='MUNICÍPIO - IBGE', 
    y='Casos', 
    hue='Período', 
    hue_order=[rotulo_estado_p1, rotulo_estado_p2],
    palette=['#326273', '#eb5e28'],
    ax=ax
)

# Adicionar rótulos com a quantidade em todas as colunas
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3, fontsize=9)

# Títulos e Eixos
ax.set_title(
    f'Comparativo de casos prováveis de Chikungunya - SE{semanas_analise} a SE{se_atual} - {nome_estado} - 2026', 
    fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Município', fontsize=16, labelpad=16)
ax.tick_params(axis='x', labelsize=10, rotation=15)  
ax.set_ylabel('Casos Prováveis', fontsize=16, labelpad=16)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)

# Legenda do Período
ax.legend(title= 'Recorte de Análise', fontsize=11, loc='upper right', frameon=True)

# RODAPÉ E SALVAMENTO
ax_rodape = grafico_municipio.add_subplot(gs[1])
ax_rodape.axis('OFF')
rodape_texto = (
    f'Fonte: SINAN | Dados até: {data_analise}\n'
    f'Observação: A quantificação dos casos deu-se a partir da variável município de residência (ID_MN_RESI).'
)
ax_rodape.text(0.0, -0.9, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, f'chik_{nome_estado}_provaveis_municipios_comparativo_4_semanas.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# MUNICÍPIOS COM MAIOR FREQUÊNCIA DE CASOS CONFIRMADOS EM UM ESTADO NAS ÚLTIMAS OITO SEMANAS

# Filtrar a base do SINAN para o estado desejado
df_gal_mun = df_gal_final_filtrado[df_gal_final_filtrado['UF'] == filtro_estado].copy()

# Tratamento dos códigos IBGE para o cruzamento
# Transformar o código IBGE de 7 dígitos do arquivo em 6 dígitos (como no SINAN)
df_municipios['COD_IBGE_6'] = df_municipios['CÓDIGO DO MUNICÍPIO - IBGE'].astype(str).str[:6]

# Garantir que a coluna do GAL também seja tratada como string
df_gal_mun['IBGE'] = df_gal_mun['IBGE'].astype(str)

# Cruzar (Merge) as duas bases
# Trazemos as informações do df_municipios para o df_sinan baseando-se no código de 6 dígitos
df_completo = pd.merge(df_gal_mun, df_municipios, left_on='IBGE', right_on='COD_IBGE_6', how='left')

# Contar os casos por município
# Contamos a frequência de cada nome na coluna 'MUNICÍPIO - IBGE'
casos_por_municipio = df_completo.groupby('MUNICÍPIO - IBGE')['casos_positivos'].sum().reset_index()
casos_por_municipio = casos_por_municipio.sort_values(by='casos_positivos', ascending= False)
casos_por_municipio.columns = ['Município', 'Total']

# Selecionar os municípios com mais casos para o gráfico não ficar ilegível
top_municipios = casos_por_municipio.head(10)

# Configurar o estilo visual
grafico_municipio = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_municipio.add_subplot(gs[0])

# Criar o Gráfico de Barras Verticais
# Ordenado do maior para o menor automaticamente pelo value_counts()
sns.barplot(data=top_municipios, x = 'Município', y = 'Total', color= "#326273", ax=ax)
ax.bar_label(ax.containers[0], fmt = '%d', padding = 3, fontsize = 10)
ax.set_title(f'Casos confirmados de chikungunya por municípios - {nome_estado} - SE{semanas_analise} a SE{se_atual} - 2026', 
          fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Município', fontsize=16, labelpad=16)
ax.tick_params(axis='x', labelsize=10, rotation=15)
ax.set_ylabel('Casos Confirmados', fontsize=16, labelpad=16)
#ax.set_ylim (0, 10000)
ax.grid(axis='y', linestyle = '--', alpha = 0.7)
ax.grid(axis='x', linestyle = '--', alpha = 0.7)

# Informações rodapé 
ax_rodape = grafico_municipio.add_subplot(gs[1])
ax_rodape.axis('OFF')
rodape_texto = (f'Fonte: GAL | Dados até: {data_gal}\n')
ax_rodape.text(0.0, -0.5, rodape_texto, ha = 'left', va = 'center', fontsize = 9, color = 'black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, f'chik_{nome_estado}_confirmados_municipios_quatro_semanas.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches = 'tight')
plt.show()

In [ ]:
# CASOS PROVÁVEIS E CONFIRMADOS (GAL) POR SEMANA EPIDEMIOLÓGICA EM UM ESTADO
# Tratamento dos Dados
df_provaveis['SE'] = pd.to_numeric(df_provaveis['SE'], errors='coerce')
df_filtrado = df_provaveis[df_provaveis['SG_UF'] == filtro_estado]

df_provaveis_semanas = df_filtrado['SE'].value_counts().reset_index()
df_provaveis_semanas.columns = ['SE', 'Total']
df_provaveis_semanas = df_provaveis_semanas.sort_values('SE').reset_index(drop=True)

df_gal_filtrado = df_gal_final_filtrado[df_gal_final_filtrado['UF'] == filtro_estado]
df_gal_semana = df_gal_filtrado.groupby('SE')[['casos_positivos']].sum().reset_index()

# União dos dois dataframe
df_merged = pd.merge(df_provaveis_semanas, df_gal_semana, on='SE', how='left').fillna(0)

# Configurar o estilo visual
grafico_sem = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_sem.add_subplot(gs[0])

# Criar o Gráfico de Barras Verticais (Usando 'color' e 'label')
sns.barplot(data=df_merged, x='SE', y='Total', color="#326273", ax=ax, label='Prováveis (SINAN)')

# Rótulos das Barras
ax.bar_label(ax.containers[0], fmt='%d', padding=3, fontsize=10)

# Criar o Gráfico de Linha
# ax.plot é mais seguro para garantir o alinhamento central com as barras do seaborn
ax.plot(df_merged.index, df_merged['casos_positivos'], color="#f77f00", linewidth=3, marker='o', markersize=6, label='Confirmados (GAL)')

# Rótulos da Linha (Loop que passa por cada ponto e escreve o número em cima)
for i, row in df_merged.iterrows():
    # Só coloca o rótulo se tiver mais de 0 casos para não poluir o gráfico
    if row['casos_positivos'] > 0: 
        # O +80 abaixo empurra o texto um pouco para cima do ponto da linha para não sobrepor
        ax.text(i, row['casos_positivos'] + 9, f"{int(row['casos_positivos'])}", 
                ha='center', va='bottom', fontsize=10, color="#dad7cd", fontweight='bold')

# Estilização do Eixo e Legenda
ax.set_title(f'Casos prováveis e confirmados de chikungunya - {nome_estado} - 2026', 
            fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Prováveis e Confirmados', fontsize=16, labelpad=16)
ax.set_xlim(-0.5, len(df_merged) - 0.5)  # Ajusta os limites do eixo x para centralizar as barras
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)

# Agora a legenda vai funcionar perfeitamente pegando os 'labels' definidos nas barras e linhas
ax.legend(fontsize=11, loc='upper right', frameon=True)

# Informações rodapé 
ax_rodape = grafico_sem.add_subplot(gs[1])
ax_rodape.axis('off')
rodape_texto = f'Fonte: SINAN e GAL | Dados até: {data_analise}\n'
ax_rodape.text(0.0, -0.0, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, f'chik_{nome_estado}_provaveis_confirmados_semana.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# COMPARATIVO ENTRE EXAMES REALIZADOS E CONFIRMADOS GAL POR SEMANA EPIDEMIOLÓGICA

# Filtragem da base para o estado desejado
df_gal = df_gal_final[df_gal_final['UF'] == filtro_estado]

# contar exames realizados na base do GAL
df_exames = df_gal.groupby('SE')['exames_realizados'].sum().reset_index()
df_exames.columns = ['SE', 'Total']

# Contar os casos positivos na base do GAL
df_conf = df_gal.groupby('SE')['casos_positivos'].sum().reset_index()
df_conf.columns = ['SE', 'Total']

df_exames['Exames'] = 'Exames Realizados'
df_conf['Exames'] = 'Exames Confirmados'

# Configurar o estilo visual
grafico_sem = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_sem.add_subplot(gs[0])

df_gal_combinado = pd.concat([df_exames, df_conf], ignore_index=True)

# Criar o Gráfico de Barras Verticais
sns.barplot(data=df_gal_combinado, x='SE', y='Total', hue='Exames',
            palette= ['#326273', '#eb5e28'], ax=ax, width= 0.9)

for c in ax.containers:
    ax.bar_label(c, fmt='%d', padding=3, fontsize=10)

# Estilização dos Eixos e Legenda
ax.set_title(f'Exames realizados e positivos para chikungunya por SE - {nome_estado} - 2026', 
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Exames Realizados e Confirmados', fontsize=16, labelpad=16)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)
ax.legend(title='', fontsize=11, title_fontsize=12, loc='upper left', frameon=True)

# Informações rodapé 
ax_rodape = grafico_sem.add_subplot(gs[1])
ax_rodape.axis('off')
rodape_texto = (f'Fonte: GAL | Dados até: {data_gal} | '
                f'Exames Realizados: {df_exames['Total'].sum():.0f} | '
                f'Exames Confirmados: {df_conf['Total'].sum():.0f}')

ax_rodape.text(0.0, 0.0, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, f'chik_{nome_estado}_realizados_confirmados_gal_semana.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# COMPARATIVO ENTRE CONFIRMADOS SINAN E CONFIRMADOS GAL
# Filtrar casos confirmados na base do SINAN
df_notificados['SE'] = pd.to_numeric(df_notificados['SE'], errors='coerce')
df_sinan_conf = df_notificados[(df_notificados['SG_UF'] == filtro_estado) & (df_notificados['CLASSI_FIN'] == 13)]
df_conf_semanas = df_sinan_conf.groupby('SE').size().reset_index(name='Total')

# Contar os casos positivos na base do GAL
df_gal_conf = df_gal_final[df_gal_final['UF'] == filtro_estado]
df_gal_conf = df_gal_conf.groupby('SE')['casos_positivos'].sum().reset_index()
df_gal_conf.columns = ['SE', 'Total']

df_conf_semanas['Confirmados'] = 'Confirmados SINAN'
df_gal_conf['Confirmados'] = 'Confirmados GAL'

# Configurar o estilo visual
grafico_sem = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_sem.add_subplot(gs[0])

df_combinado = pd.concat([df_conf_semanas, df_gal_conf], ignore_index=True)

# Criar o Gráfico de Barras Verticais
sns.barplot(data=df_combinado, x='SE', y='Total', hue='Confirmados',
            palette= ['#326273', '#eb5e28'], ax=ax, width= 0.9)

for c in ax.containers:
    ax.bar_label(c, fmt='%d', padding=3, fontsize=10)

# Estilização dos Eixos e Legenda
ax.set_title(f'Casos confirmados de chikungunya por SE - GAL e SINAN - {nome_estado} - 2026', 
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Confirmados', fontsize=16, labelpad=16)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)
ax.legend(title='', fontsize=11, title_fontsize=12, loc='upper right', frameon=True)

# Informações rodapé 
ax_rodape = grafico_sem.add_subplot(gs[1])
ax_rodape.axis('off')
rodape_texto = (f'Fonte: SINAN e GAL | Dados até: {data_analise} | '
                f'Confirmados SINAN: {df_conf_semanas['Total'].sum():.0f} | '
                f'Confirmados GAL: {df_gal_conf['Total'].sum():.0f}')

ax_rodape.text(0.0, 0.0, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, f'chik_{nome_estado}_confirmados_sinan_gal.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# CASOS CONFIRMADOS POR CRITÉRIO DE CONFIRMAÇÃO APLICADO
# Filtrar casos confirmados por critério laboratorial na base do SINAN
df_notificados['SE'] = pd.to_numeric(df_notificados['SE'], errors='coerce')
df_conf_criterio = df_notificados[df_notificados['SG_UF'] == filtro_estado]

df_conf_lab = df_conf_criterio[(df_conf_criterio['CLASSI_FIN'] == 13) & (df_conf_criterio['CRITERIO'] == 1)]['SE'].value_counts().reset_index()
df_conf_lab.columns = ['SE', 'Total']
df_conf_lab = df_conf_lab.sort_values('SE').reset_index(drop=True)

# Filtrar casos confirmados por criterio clini-epidemiologico na base do SINAN
df_conf_clep = df_conf_criterio[(df_conf_criterio['CLASSI_FIN'] == 13) & (df_conf_criterio['CRITERIO'] == 2)]['SE'].value_counts().reset_index()
df_conf_clep.columns = ['SE', 'Total']
df_conf_clep = df_conf_clep.sort_values('SE').reset_index(drop=True)

df_conf_lab['Confirmados'] = 'Laboratorial'
df_conf_clep['Confirmados'] = 'Clinico-Epidemiológico'

# Configurar o estilo visual
grafico_sem = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_sem.add_subplot(gs[0])

df_combinado = pd.concat([df_conf_lab, df_conf_clep], ignore_index=True)

# Criar o Gráfico de Barras Verticais
sns.barplot(data=df_combinado, x='SE', y='Total', hue='Confirmados',
            palette= ['#326273', '#eb5e28'], ax=ax, width= 0.9)

for c in ax.containers:
    ax.bar_label(c, fmt='%d', padding=3, fontsize=10)

# Estilização dos Eixos e Legenda
ax.set_title(f'Casos confirmados de chikungunya por SE e critério de confirmação - {nome_estado} - 2026', 
             fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Confirmados', fontsize=16, labelpad=16)
ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)
ax.legend(title='', fontsize=11, title_fontsize=12, loc='upper left', frameon=True)

# Informações rodapé 
ax_rodape = grafico_sem.add_subplot(gs[1])
ax_rodape.axis('off')
rodape_texto = (f'Fonte: SINAN | Dados até: {data_analise} | '
                f'Confirmados - Laboratorial: {df_conf_lab['Total'].sum():.0f} | '
                f'Confirmados - Clinico-Epidemiológico: {df_conf_clep['Total'].sum():.0f}')

ax_rodape.text(0.0, 0.0, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, f'chik_{nome_estado}_confirmados_criterio.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# CLASSIFICAÇÃO FINAL POR SEMANA EPIDEMIOLÓGICA
# Tratamento da Classificação
# Transformar em número. O que for vazio ('') vira NaN (Nulo)
df_notificados['CLASSI_FIN'] = pd.to_numeric(df_notificados['CLASSI_FIN'], errors='coerce')
df_notif_filtrado = df_notificados[df_notificados['SG_UF'] == filtro_estado]

# Preencher os valores nulos (NaN) com um código numérico fictício (ex: 99) 
# Isso impede que o groupby ignore os casos "Em Investigação"
df_notif_filtrado['CLASSI_FIN'] = df_notif_filtrado['CLASSI_FIN'].fillna(99)

# Novo dicionário com o código 99 representando o que estava vazio
mapa_class = {5: 'Descartado', 13: 'Confirmado', 99: 'Em Investigação'}

# Agrupar os dados
# Dataframe para o gráfico de barras empilhadas e tabela
df_class = df_notif_filtrado.groupby(['SE', 'CLASSI_FIN']).size().unstack(fill_value=0) 
df_class = df_class.rename(columns=mapa_class)

# Configurar o estilo visual
grafico_class = plt.figure(figsize=(15, 8))
sns.set_theme(style="whitegrid")

gs = gridspec.GridSpec(2, 1, height_ratios=[12, 1])
ax = grafico_class.add_subplot(gs[0])

# Criar o Gráfico de Barras Empilhadas
# Adicionei cores distintas para cada classificação
cores = {'Confirmado': '#326273', 'Descartado': '#5c9ead', 'Em Investigação': '#eb5e28'}
df_class.plot(kind='bar', stacked=True, ax=ax, width=0.8, color=[cores.get(col, '#333333') for col in df_class.columns])

# Adicionar os rótulos de dados (bar_label)
# Como são barras empilhadas, precisamos passar por CADA container (cada pedaço da barra)
for c in ax.containers:
    # Condição para só escrever o número se ele for maior que 0 (evita poluir o gráfico com zeros)
    labels = [f'{v.get_height():.0f}' if v.get_height() > 0 else '' for v in c]
    # label_type='center' coloca o número bem no meio da cor respectiva
    ax.bar_label(c, labels=labels, label_type='center', fontsize=9, color='white', fontweight='bold')

ax.set_title(f'Classificação final dos casos notificados de chikungunya por SE - {nome_estado} - 2026', 
          fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Semana Epidemiológica', fontsize=16, labelpad=16)
ax.set_ylabel('Casos Notificados', fontsize=16, labelpad=16)

# O eixo X no pandas fica rotacionado em 90 graus por padrão. Isso conserta e deixa reto:
plt.xticks(rotation=0) 

# Ajustar o limite do eixo Y (adapte se necessário)
# ax.set_ylim(0, 1400) # Removi temporariamente para o pandas ajustar automaticamente às barras somadas

# Melhorar a legenda
ax.legend(title='Classificação Final', fontsize=11, title_fontsize=12, loc='upper right')

ax.grid(axis='y', linestyle='--', alpha=0.7)
ax.grid(axis='x', linestyle='--', alpha=0.7)

# Informações rodapé 
ax_rodape = grafico_class.add_subplot(gs[1])
ax_rodape.axis('off') # Minúsculo
rodape_texto = f'Fonte: SINAN | Dados até: {data_analise}'
# Ajustado para ficar centralizado verticalmente no rodapé
ax_rodape.text(0.0, 0.0, rodape_texto, ha='left', va='center', fontsize=9, color='black')

# Finalização e Salvamento
salvar_pasta = os.path.join(local_pasta, f'chik_{nome_estado}_provaveis_classificacao.png')
plt.savefig(salvar_pasta, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# SALVAR GRÁFICOS EM PDF ÚNICO
def gerar_pdf(lista_arquivos, nome_pdf: f'Relatorio_Chikungunya_{nome_estado}_2026.pdf'):
    c = canvas.Canvas(nome_pdf, pagesize=A4)
    largura, altura = A4
    margem = 30
    
    # Função auxiliar para escrever o cabeçalho em todas as páginas
    def desenhar_cabecalho(canvas):
        canvas.setFont("Helvetica-Bold", 16)
        canvas.drawCentredString(largura / 2, altura - 25, f"Análise Chikungunya - {nome_estado} - 2026")       
        canvas.line(margem, altura - 35, largura - margem, altura - 35)                                         

    def processar_paginas(imagens_lista):
        espacamento = 5                                 
        area_h = (altura / 2) - (margem + 40)   
        largura_max = largura - (2 * margem)    
    
        def desenhar_figura(img_path, pos_y):
            # Verifica se a imagem existe antes de desenhar (evita o erro OSError)
            if os.path.exists(img_path):
                c.drawImage(img_path, margem, pos_y, width=largura_max, height=area_h, preserveAspectRatio=True)
            else:
                print(f"⚠️ Aviso: A imagem {img_path} não foi encontrada.")

        def desenhar_rodape(canvas, texto):
            canvas.setFont("Helvetica-Oblique", 5) 
            canvas.drawString(30, 20, texto)

        texto_rodape = 'Análise realizada por Coordenação-Geral de Vigilância das Arboviroses (CGARB/DEDT/SVSA/MS)'

        # 1. PÁGINA 1
        desenhar_cabecalho(c)
        desenhar_figura(imagens_lista[0], (altura / 2) + espacamento) 
        desenhar_figura(imagens_lista[1], margem)
        desenhar_rodape(c, texto_rodape) # Rodapé da Página 1
        c.showPage()
    
        # 2. PÁGINAS SEGUINTES
        restante = imagens_lista[2:]
        for i in range(0, len(restante), 2):
            desenhar_cabecalho(c)
            grupo = restante[i:i+2]
        
            pos_y_lista = [(altura / 2) + espacamento, margem]
        
            for idx, img_path in enumerate(grupo):
                desenhar_figura(img_path, pos_y_lista[idx])
            
            desenhar_rodape(c, texto_rodape) # CORREÇÃO: Rodapé nas outras páginas!
            c.showPage()

    processar_paginas(lista_arquivos)
    c.save()
    print(f"Arquivo {os.path.basename(nome_pdf)} gerado com sucesso e salvo na pasta!")

# Lista dos arquivos
arquivos = [
    f'chik_{nome_estado}_provaveis_municipios.png',
    f'chik_{nome_estado}_provaveis_municipios_quatro_semanas.png',
    f'chik_{nome_estado}_provaveis_municipios_comparativo_4_semanas.png',
    f'chik_{nome_estado}_confirmados_municipios_quatro_semanas.png',
    f'chik_{nome_estado}_provaveis_confirmados_semana.png',
    f'chik_{nome_estado}_confirmados_sinan_gal.png',
    f'chik_{nome_estado}_confirmados_criterio.png',
    f'chik_{nome_estado}_provaveis_classificacao.png',
]

graficos = [os.path.join(local_pasta, nome) for nome in arquivos]
salvar_pdf = os.path.join(local_pasta, f'Relatorio_Chikungunya_{nome_estado}_2026.pdf')

gerar_pdf(graficos, nome_pdf=salvar_pdf)

#### Algoritmo desenvolvido por Dengue Brazil